# 06 · Suitability and validation (Phase 7)

1. the combined score and its classes S1–S3 / N (P7.1);
2. candidate sites (P7.2);
3. the validation harness: every score tested the same way against growth from 2018 to 2022/23 (P7.4, P7.5);
4. the MCDA / AHP baseline (P7.3);
5. sensitivity (P7.6).

Build: `python -m src.pipeline --from-stage suitability`. Checked by `scripts/verify_suitability.py`, `scripts/verify_mcda.py`, `scripts/verify_sensitivity.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.config import load_config

pd.set_option("display.max_colwidth", 90)
cfg = load_config()
MET = Path(cfg.paths["metrics"])
FIG = Path(cfg.paths["figures"])


def show(name, width=760):
    display(Image(filename=str(FIG / f"{name}.png"), width=width))

## 1. Combined score

Weighted mean of the within-run percentiles of `similarity_knn` and `rf_change`. The weight is chosen on the validation run's **training** label (built in 2020/21), never on the 2022/23 test label, and frozen for the final run.

In [ ]:
s = json.loads((MET / "suitability.json").read_text())
print("weight: similarity", s["w_similarity"], "/ change RF", s["w_rf_change"])
pd.DataFrame(s["weight_table"]).set_index("w_similarity").round(4)

In [ ]:
pd.DataFrame({k.replace("class_counts_", ""): v for k, v in s.items() if k.startswith("class_counts")})

In [ ]:
show("map_suitability_classes", 640)

## 2. Candidate sites

S1 cells merged into patches of edge-adjacent cells. Half the patches are single 1-ha cells, so the top sites are the best patches of at least 5 ha.

In [ ]:
sites = pd.read_csv(Path(cfg.paths["outputs"]) / "candidates.csv")
print(len(sites), "patches,", round(sites["area_ha"].sum()), "ha;", int(sites["agricultural"].sum()), "mostly cropland")
cols = ["top_site_rank", "area_ha", "mean_score", "slope_deg", "dist_major_road_m", "dist_built_m", "dominant_cover", "lon", "lat"]
sites.dropna(subset=["top_site_rank"]).sort_values("top_site_rank")[cols].round(2)

## 3. Validation

12,777 candidate cells (non-built in 2018 and 2019), 1,125 of which were built in both 2022 and 2023. Every score is as of 2018; trained models are scored out of fold.

In [ ]:
v = json.loads((MET / "validation.json").read_text())
cols = ["auc", "precision_top10", "lift_top10", "lift_top30", "auc_adjacent", "auc_outlying"]
pd.DataFrame(v["models"]).T[cols].astype(float).sort_values("auc", ascending=False).round(3)

In [ ]:
m = v["models"]["suitability"]
print(f"S1 + S2 hold {m['s1_s2_share_of_cells']:.0%} of the candidate cells and {m['s1_s2_hit_rate']:.0%} of the growth "
      f"(precision {m['s1_s2_precision']:.1%}, lift {m['s1_s2_lift']:.2f})")

In [ ]:
show("chart_roc", 520)
show("chart_toc", 520)

In [ ]:
show("chart_growth_type")

**Growth type.** Every model finds growth next to existing built-up land well (AUC ~0.8) but growth away from it at chance level (~0.5). That matches the literature: diffusional growth is much harder to predict (Wang et al. 2021). Only 91 of the 1,125 grown cells are outlying.

### Ablation

In [ ]:
a = v["ablation"]
pd.DataFrame({g: {"dropped": ", ".join(r["dropped"]), "change RF": r["rf_change"]["delta"],
                  "similarity": r["similarity_knn"]["delta"]} for g, r in a["drop"].items()}).T

In [ ]:
show("chart_ablation")

Built-up land nearby carries the signal for both models; roads and water add almost nothing once built-up context is known. Land cover nearby helps the RF but not the similarity.

## 4. MCDA / AHP baseline

In [ ]:
from src.suitability import mcda

A = pd.DataFrame(mcda.parse_matrix(cfg["mcda"]["ahp"]), index=cfg["mcda"]["criteria"], columns=cfg["mcda"]["criteria"])
w, cons = mcda.weights(cfg)
display(A.round(2))
print("weights:", w.round(3).to_dict())
print(f"lambda_max {cons['lambda_max']:.3f}, consistency ratio {cons['consistency_ratio']:.4f} (< 0.1)")

In [ ]:
mm = json.loads((MET / "mcda.json").read_text())
pd.Series(mm["criterion_auc"]).round(3).to_frame("AUC on grew, criterion alone").sort_values(by="AUC on grew, criterion alone", ascending=False)

Only distance to built-up carries real signal on its own (0.763); roads, water and land cover are close to chance. The AHP weights spread importance over all five, which dilutes it: the overlay scores 0.636. The weights are **not** tuned on the test label; this is what a fixed expert overlay does.

## 5. Sensitivity

In [ ]:
sens = json.loads((MET / "sensitivity.json").read_text())
cols = ["n_candidates", "n_grew", "auc_rf_change", "auc_dist_to_built", "auc_mcda"]
display(pd.DataFrame(sens["slope"]).set_index("slope_max_deg")[["n_excl_slope", *cols]].round(3))
display(pd.DataFrame(sens["cell"]).set_index("cell_size_m")[["n_cells", *cols]].round(3))
display(pd.DataFrame(sens["k"]).set_index("k").drop(columns="cluster_rules").round(3))

In [ ]:
show("chart_sensitivity")

**Reading it.** The slope threshold and k barely move anything (AUC within 0.01; ESRI agreement 96.6–98.8 %). Cell size changes the AUCs by at most 0.02 (the growth label itself changes with the cell, and at 200 m only 148 cells grew). The ranking change RF > distance to built-up > MCDA holds everywhere.